In [1]:
suppressPackageStartupMessages(library(MouseGastrulationData))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(Matrix))

In [2]:
getwd()

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/12_Eomes_T_Mixl1/T_E75"

In [4]:
io = list()
io$outdir = '/rds/project/rds-SDzz0CATGms/users/bt392/12_Eomes_T_Mixl1/T_E75/data/'
io$genemetadata = "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/Mmusculus_genes_BioMart.87.txt"

dir.create(io$outdir, recursive = T, showWarnings = F)

In [6]:
T_E85.sce = TChimeraData(type = c("processed"), samples = c(11:16))

snapshotDate(): 2022-10-31

see ?MouseGastrulationData and browseVignettes('MouseGastrulationData') for documentation

loading from cache

see ?MouseGastrulationData and browseVignettes('MouseGastrulationData') for documentation

downloading 1 resources

retrieving 1 resource

loading from cache

see ?MouseGastrulationData and browseVignettes('MouseGastrulationData') for documentation

downloading 1 resources

retrieving 1 resource

loading from cache

see ?MouseGastrulationData and browseVignettes('MouseGastrulationData') for documentation

downloading 1 resources

retrieving 1 resource

loading from cache

see ?MouseGastrulationData and browseVignettes('MouseGastrulationData') for documentation

downloading 1 resources

retrieving 1 resource

loading from cache

see ?MouseGastrulationData and browseVignettes('MouseGastrulationData') for documentation

downloading 1 resources

retrieving 1 resource

loading from cache

see ?MouseGastrulationData and browseVignettes('MouseGastrulationD

In [7]:
T_E85.sce

class: SingleCellExperiment 
dim: 29453 6173 
metadata(0):
assays(1): counts
rownames(29453): ENSMUSG00000051951 ENSMUSG00000089699 ...
  ENSMUSG00000095742 tomato-td
rowData names(2): ENSEMBL SYMBOL
colnames(6173): cell_31678 cell_31679 ... cell_37849 cell_37850
colData names(13): cell barcode ... somite.subct.mapped sizeFactor
reducedDimNames(2): pca.corrected.E7.5 pca.corrected.E8.5
mainExpName: NULL
altExpNames(0):

In [8]:
gene_metadata = fread(io$genemetadata)[, c('ens_id', 'symbol')] %>%
    rbind(., data.table(ens_id='tomato-td', symbol='tomato-td')) %>% 
    .[ens_id %in% rownames(T_E85.sce)] %>% 
    .[symbol!='']

T_E85.sce = T_E85.sce[rownames(T_E85.sce) %in% gene_metadata$ens_id]

In [9]:
rownames(T_E85.sce) = gene_metadata[match(rownames(T_E85.sce), ens_id), symbol]

In [10]:
summary(rownames(T_E85.sce)[order(rownames(T_E85.sce))] == gene_metadata$symbol[order(gene_metadata$symbol)])

   Mode    TRUE 
logical   27995 

In [11]:
meta = as.data.table(colData(T_E85.sce), keep.rownames=F) #%>% setnames('rn', 'cell')

Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”


In [12]:
head(meta)

cell,barcode,sample,stage,tomato,pool,stage.mapped,celltype.mapped,closest.cell,doub.density,trajectory.mapped,somite.subct.mapped,sizeFactor
<chr>,<chr>,<int>,<chr>,<lgl>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>
cell_31678,AAACCTGAGTACCGGA,11,E7.5,TRUE,6,E7.5,Nascent mesoderm,cell_48913,0.09574493,post,NA,2.641880
cell_31679,AAACCTGCATAGAAAC,11,E7.5,TRUE,6,E7.5,Def. endoderm,cell_50344,1.18459081,none,NA,2.407283
cell_31680,AAACGGGTCCCTTGTG,11,E7.5,TRUE,6,E7.0,Primitive Streak,cell_78713,0.07540544,nmp,NA,1.286243
cell_31681,AAAGCAAAGCCGCCTA,11,E7.5,TRUE,6,E7.0,Nascent mesoderm,cell_85650,0.08368138,post,NA,1.898547
cell_31682,AAAGCAACATGCCTAA,11,E7.5,TRUE,6,E7.0,Epiblast,cell_43234,0.03161075,none,NA,2.764880
cell_31683,AAAGCAAGTACGACCC,11,E7.5,TRUE,6,E7.0,Nascent mesoderm,cell_103623,0.10193134,post,NA,1.614795


In [18]:
table(meta$celltype.mapped, meta$tomato)

                                
                                 FALSE TRUE
  Allantois                          8    0
  Anterior Primitive Streak         33  110
  Blood progenitors 1               45   35
  Blood progenitors 2               59   52
  Cardiomyocytes                     1    0
  Caudal epiblast                   97  134
  Caudal Mesoderm                   27   22
  Caudal neurectoderm                9    2
  Def. endoderm                     50   77
  Doublet                          135  245
  Endothelium                        1    3
  Epiblast                         327  501
  Erythroid1                         2    6
  ExE ectoderm                     533    3
  ExE endoderm                     117    2
  ExE mesoderm                      29   24
  Gut                               65  118
  Haematoendothelial progenitors   127  125
  Intermediate mesoderm             51   34
  Mesenchyme                       236  279
  Mixed mesoderm                   112  136

In [13]:
table(meta$sample, meta$tomato)

    
     FALSE TRUE
  11     0  939
  12   773    0
  13     0  737
  14   798    0
  15     0 1444
  16  1482    0

In [15]:
table(meta$sample, meta$pool)

    
        6    7    8
  11  939    0    0
  12  773    0    0
  13    0  737    0
  14    0  798    0
  15    0    0 1444
  16    0    0 1482

In [14]:
lapply(unique(meta$sample), function(x){ # unique(meta$sample)
    tmp = meta[sample==x]
    tmp.sce = T_E85.sce[,tmp$cell]
    
    tmp.outdir = paste0(io$outdir, 'sample_', x, '/outs/filtered_feature_bc_matrix')
    dir.create(tmp.outdir, recursive = TRUE, showWarnings = FALSE)
    fwrite(tmp[,'barcode'], file.path(tmp.outdir, 'barcodes.tsv.gz'), col.names=F)
    
    features =  gene_metadata[match(rownames(tmp.sce), symbol), c('ens_id', 'symbol')] %>%
        .[, gene_expression := 'Gene Expression']
    stopifnot(features$symbol == rownames(tmp.sce))
    fwrite(features, file.path(tmp.outdir, 'features.tsv.gz'), col.names=F)

    writeMM(counts(tmp.sce), file.path(tmp.outdir, 'matrix.mtx.gz'))
    cat(paste0('Sample ', x, ' done \n'))
})

Sample 11 done 
Sample 12 done 
Sample 13 done 
Sample 14 done 
Sample 15 done 
Sample 16 done 


[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

[[4]]
NULL

[[5]]
NULL

[[6]]
NULL